# Lectures 6–7: Projectors & QR Factorization
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Lecture 6: Projectors

### Idempotence: $P^2 = P$

In [ ]:
# Example 1: projection onto the x-axis
P1 = np.array([[1, 0],
               [0, 0]], dtype=float)

v = np.array([3, 5])
print(f"P1 @ v = {P1 @ v}")
print(f"P1^2 == P1: {np.allclose(P1 @ P1, P1)}")

# Example 2: projection onto the line x1 = x2
P2 = 0.5 * np.array([[1, 1],
                      [1, 1]], dtype=float)

print(f"\nP2 @ v = {P2 @ v}")
print(f"P2^2 == P2: {np.allclose(P2 @ P2, P2)}")

### Orthogonal vs. Oblique Projectors

Both project onto the line $y = x$, but the orthogonal one drops perpendicular to the line while the oblique one drops along the $x$-axis.

In [ ]:
P_orth = 0.5 * np.array([[1, 1], [1, 1]], dtype=float)
P_oblq = np.array([[0, 1], [0, 1]], dtype=float)

v = np.array([3.0, 1.0])

Pv_orth = P_orth @ v
Pv_oblq = P_oblq @ v

print("Orthogonal projector:")
print(f"  P_orth symmetric: {np.allclose(P_orth, P_orth.T)}")
print(f"  P_orth @ v = {Pv_orth}, residual = {v - Pv_orth}")

print("\nOblique projector:")
print(f"  P_oblq symmetric: {np.allclose(P_oblq, P_oblq.T)}")
print(f"  P_oblq @ v = {Pv_oblq}, residual = {v - Pv_oblq}")

In [ ]:
# Visualize both projectors
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, P, title in zip(axes,
    [P_orth, P_oblq],
    ['Orthogonal projector', 'Oblique projector']):

    Pv = P @ v
    # Target line y = x
    ax.plot([-4, 4], [-4, 4], 'k--', alpha=0.3, label='$y = x$')
    # v
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    ax.text(v[0]+0.1, v[1]+0.2, '$v$', fontsize=12, color='blue')
    # Pv
    ax.annotate('', xy=Pv, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='green', lw=2))
    ax.text(Pv[0]-0.6, Pv[1]+0.2, '$Pv$', fontsize=12, color='green')
    # residual
    ax.annotate('', xy=v, xytext=Pv,
                arrowprops=dict(arrowstyle='->', color='red', lw=2, linestyle='--'))
    ax.text((v[0]+Pv[0])/2+0.15, (v[1]+Pv[1])/2-0.3, 'residual', fontsize=10, color='red')

    ax.set_xlim(-1, 4.5)
    ax.set_ylim(-1, 4.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Visualizing an Oblique Projector's Map

The non-orthogonal projector $P = \begin{pmatrix}0&0\\1&1\end{pmatrix}$ maps every point to the line $x_1 = 0$, $x_2 = x_1 + x_2$. Motion is not perpendicular to the range.

In [ ]:
P = np.array([[0, 0], [1, 1]], dtype=float)

print(f"P^2 == P: {np.allclose(P @ P, P)}")
print(f"Symmetric: {np.allclose(P, P.T)}")
print(f"Singular values: {np.linalg.svd(P, compute_uv=False).round(4)}")

# Grid of points and their images
xs = np.arange(-4, 5)
X, Y = np.meshgrid(xs, xs)
pts = np.column_stack([X.ravel(), Y.ravel()])  # (N, 2)
Ppts = (P @ pts.T).T

plt.figure(figsize=(6, 6))
for i in range(len(pts)):
    plt.plot([pts[i, 0], Ppts[i, 0]], [pts[i, 1], Ppts[i, 1]], 'b-', alpha=0.3)
plt.plot(Ppts[:, 0], Ppts[:, 1], 'ko', markersize=3, label='$Px$')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Oblique projector: motion is NOT perpendicular to range')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### Constructing Orthogonal Projectors via $P = \hat{Q}\hat{Q}^T$

In [ ]:
# Project onto the xy-plane in R^3
Q_hat = np.array([[1, 0],
                   [0, 1],
                   [0, 0]], dtype=float)

P = Q_hat @ Q_hat.T
print("P (project onto xy-plane):")
print(P)

v = np.array([3, 4, 5])
print(f"\nPv = {P @ v}  (z-component killed)")
print(f"P^2 == P: {np.allclose(P @ P, P)}")
print(f"P^T == P: {np.allclose(P, P.T)}")

In [ ]:
# Project onto an arbitrary plane in R^3
# Plane spanned by two vectors (not orthonormal)
a1 = np.array([1, 1, 0], dtype=float)
a2 = np.array([0, 1, 1], dtype=float)
A = np.column_stack([a1, a2])

# Orthonormalize first, then form projector
Q_hat, _ = np.linalg.qr(A)
P = Q_hat @ Q_hat.T

v = np.array([3, 4, 5], dtype=float)
Pv = P @ v
residual = v - Pv
print(f"v = {v}")
print(f"Pv = {Pv.round(6)}")
print(f"Residual = {residual.round(6)}")
print(f"Residual perpendicular to range: {np.allclose(Q_hat.T @ residual, 0)}")

### Rank-1 Projector and Complementary Projector

In [ ]:
q = np.array([1, 1], dtype=float) / np.sqrt(2)
P = np.outer(q, q)       # project onto line through q
Pc = np.eye(2) - P       # complementary projector

v = np.array([5, 3], dtype=float)
Pv = P @ v
Pcv = Pc @ v

print(f"P  = \n{P}")
print(f"Pv  = {Pv}")
print(f"(I-P)v = {Pcv}")
print(f"Pv + (I-P)v = {Pv + Pcv}  (should equal v = {v})")
print(f"Pv · (I-P)v = {Pv @ Pcv:.10f}  (should be 0)")

In [ ]:
# Visualize the decomposition v = Pv + (I-P)v
plt.figure(figsize=(6, 6))
origin = [0, 0]

# Line through q
plt.plot([-5, 5], [-5, 5], 'k--', alpha=0.3, label='$y=x$ (range of $P$)')
plt.plot([-5, 5], [5, -5], 'k:', alpha=0.3, label='$y=-x$ (range of $I-P$)')

plt.annotate('', xy=v, xytext=origin,
             arrowprops=dict(arrowstyle='->', color='blue', lw=2))
plt.text(v[0]+0.1, v[1]+0.2, '$v$', fontsize=13, color='blue')

plt.annotate('', xy=Pv, xytext=origin,
             arrowprops=dict(arrowstyle='->', color='green', lw=2))
plt.text(Pv[0]-0.8, Pv[1]+0.2, '$Pv$', fontsize=13, color='green')

plt.annotate('', xy=Pcv, xytext=origin,
             arrowprops=dict(arrowstyle='->', color='red', lw=2))
plt.text(Pcv[0]+0.1, Pcv[1]-0.4, '$(I-P)v$', fontsize=13, color='red')

# Show the addition
plt.annotate('', xy=v, xytext=Pv,
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5, linestyle='--'))

plt.xlim(-2, 6)
plt.ylim(-2, 6)
plt.gca().set_aspect('equal')
plt.grid(True, alpha=0.3)
plt.title('Decomposition: $v = Pv + (I-P)v$', fontsize=13)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Properties of Orthogonal Projectors

In [ ]:
# Build a rank-2 orthogonal projector in R^5
np.random.seed(7)
Q_hat, _ = np.linalg.qr(np.random.randn(5, 2))
P = Q_hat @ Q_hat.T

eigvals = np.linalg.eigvalsh(P)
print(f"Eigenvalues: {np.sort(eigvals).round(10)}")
print(f"Trace:       {np.trace(P):.1f}  (should equal rank = 2)")
print(f"||P||_2:     {np.linalg.norm(P, 2):.4f}  (should be 1)")
print(f"P^2 == P:    {np.allclose(P @ P, P)}")
print(f"P^T == P:    {np.allclose(P, P.T)}")

---
## Lecture 7: QR Factorization

### Gram-Schmidt Step by Step

In [ ]:
# Concrete 3x2 example from the lecture
A = np.array([[1, 1],
              [0, 1],
              [1, 0]], dtype=float)

m, n = A.shape
Q = np.zeros((m, n))
R = np.zeros((n, n))

# Step 1: normalize a1
R[0, 0] = np.linalg.norm(A[:, 0])
Q[:, 0] = A[:, 0] / R[0, 0]
print(f"Step 1: q1 = {Q[:, 0].round(6)}, r11 = {R[0,0]:.6f}")

# Step 2: remove q1 component from a2, normalize
R[0, 1] = Q[:, 0] @ A[:, 1]
v = A[:, 1] - Q[:, 0] * R[0, 1]
R[1, 1] = np.linalg.norm(v)
Q[:, 1] = v / R[1, 1]
print(f"Step 2: q2 = {Q[:, 1].round(6)}, r12 = {R[0,1]:.6f}, r22 = {R[1,1]:.6f}")

print(f"\nQ =\n{Q.round(6)}")
print(f"\nR =\n{R.round(6)}")
print(f"\nQ^T Q = I: {np.allclose(Q.T @ Q, np.eye(n))}")
print(f"QR = A:    {np.allclose(Q @ R, A)}")

### Classical Gram-Schmidt (general loop)

In [ ]:
def classical_gram_schmidt(A):
    """Reduced QR factorization via classical Gram-Schmidt."""
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    R[0, 0] = np.linalg.norm(A[:, 0])
    Q[:, 0] = A[:, 0] / R[0, 0]
    for j in range(1, n):
        R[:j, j] = Q[:, :j].T @ A[:, j]
        v = A[:, j] - Q[:, :j] @ R[:j, j]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

# Test on the 6x4 example from the lecture
np.random.seed(0)
A = np.round(10 * np.random.rand(6, 4))
print(f"A =\n{A}\n")

Q, R = classical_gram_schmidt(A)
print(f"||Q^T Q - I|| = {np.linalg.norm(Q.T @ Q - np.eye(4)):.2e}")
print(f"||A - QR||    = {np.linalg.norm(A - Q @ R):.2e}")
print(f"\nR =\n{R.round(6)}")

### Comparison with NumPy's QR

In [ ]:
Q_np, R_np = np.linalg.qr(A)
print(f"NumPy Q shape: {Q_np.shape}  (reduced by default)")
print(f"NumPy R shape: {R_np.shape}")
print(f"||A - Q_np R_np|| = {np.linalg.norm(A - Q_np @ R_np):.2e}")

# Sign convention: NumPy forces positive diagonal in R
print(f"\nDiagonal of our R:    {np.diag(R).round(4)}")
print(f"Diagonal of NumPy R:  {np.diag(R_np).round(4)}")

### Full vs. Reduced QR

In [ ]:
A = np.array([[1, 1],
              [0, 1],
              [1, 0]], dtype=float)

# Reduced
Q_red, R_red = np.linalg.qr(A, mode='reduced')
print(f"Reduced Q shape: {Q_red.shape}, R shape: {R_red.shape}")

# Full
Q_full, R_full = np.linalg.qr(A, mode='complete')
print(f"Full Q shape:    {Q_full.shape}, R shape: {R_full.shape}")

print(f"\nFull Q =\n{Q_full.round(6)}")
print(f"\nFull R =\n{R_full.round(6)}")

q3 = Q_full[:, 2]
print(f"\nq3 (left nullspace) = {q3.round(6)}")
print(f"q3 · q1 = {q3 @ Q_full[:, 0]:.1e}")
print(f"q3 · q2 = {q3 @ Q_full[:, 1]:.1e}")
print(f"A^T q3  = {(A.T @ q3).round(10)}  (should be zero)")

### Reading the R Factor

In [ ]:
np.random.seed(3)
A = np.random.randn(5, 3)
Q, R = np.linalg.qr(A)

print("R (reduced):")
print(R.round(4))
print()
for j in range(3):
    print(f"Column {j+1}:")
    print(f"  |r_{j+1}{j+1}| = {abs(R[j,j]):.4f}  (new information in a_{j+1})")
    for i in range(j):
        print(f"  r_{i+1}{j+1} = {R[i,j]:.4f}  (component of a_{j+1} along q_{i+1})")

### Solving Least Squares via QR

Fit a line $y = c_1 + c_2 t$ through $(0,1)$, $(1,2)$, $(2,4)$.

In [ ]:
t_data = np.array([0, 1, 2], dtype=float)
b = np.array([1, 2, 4], dtype=float)

A = np.column_stack([np.ones_like(t_data), t_data])
print(f"A =\n{A}\n")

# QR approach
Q_hat, R_hat = np.linalg.qr(A)
c_qr = np.linalg.solve(R_hat, Q_hat.T @ b)
print(f"Least squares coefficients (QR):     c = {c_qr.round(6)}")

# Compare with normal equations
c_ne = np.linalg.solve(A.T @ A, A.T @ b)
print(f"Least squares coefficients (normal): c = {c_ne.round(6)}")

# Compare with np.linalg.lstsq
c_lstsq, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
print(f"Least squares coefficients (lstsq):  c = {c_lstsq.round(6)}")

In [ ]:
# Plot the fit
t_fine = np.linspace(-0.5, 2.5, 100)
y_fit = c_qr[0] + c_qr[1] * t_fine

plt.figure(figsize=(7, 4.5))
plt.plot(t_data, b, 'ro', markersize=8, label='Data')
plt.plot(t_fine, y_fit, 'b-', linewidth=2,
         label=f'$y = {c_qr[0]:.2f} + {c_qr[1]:.2f}t$')

# Residuals
for ti, bi in zip(t_data, b):
    yi = c_qr[0] + c_qr[1] * ti
    plt.plot([ti, ti], [yi, bi], 'r--', alpha=0.5)

plt.xlabel('$t$')
plt.ylabel('$y$')
plt.title('Least squares line via QR factorization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Why QR Beats the Normal Equations: Condition Number

In [ ]:
np.random.seed(10)
m, n = 50, 5
A = np.random.randn(m, n)

kappa_A = np.linalg.cond(A)
kappa_AtA = np.linalg.cond(A.T @ A)

print(f"cond(A)    = {kappa_A:.2f}")
print(f"cond(A^TA) = {kappa_AtA:.2f}")
print(f"Ratio:       {kappa_AtA / kappa_A:.2f}  (approx cond(A))")
print(f"\nNormal equations square the condition number!")
print(f"QR avoids this by working with A directly.")

### Back Substitution

In [ ]:
def back_substitution(R, z):
    """Solve Rx = z where R is upper triangular."""
    n = len(z)
    x = np.zeros(n)
    x[n-1] = z[n-1] / R[n-1, n-1]
    for i in range(n-2, -1, -1):
        x[i] = (z[i] - R[i, i+1:] @ x[i+1:]) / R[i, i]
    return x

# Solve Ax = b via QR + back substitution
np.random.seed(5)
n = 9
A = np.random.randn(n, n)
b = np.arange(1, n+1, dtype=float)

Q, R = np.linalg.qr(A)
z = Q.T @ b
x = back_substitution(R, z)

print(f"x = {x.round(6)}")
print(f"||Ax - b|| = {np.linalg.norm(A @ x - b):.2e}")

# Compare with direct solve
x_direct = np.linalg.solve(A, b)
print(f"||x - x_direct|| = {np.linalg.norm(x - x_direct):.2e}")

### Visualizing Gram-Schmidt in $\mathbb{R}^3$

In [ ]:
A = np.array([[1, 1],
              [0, 1],
              [1, 0]], dtype=float)

Q, R = np.linalg.qr(A)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

origin = [0, 0, 0]
colors_a = ['steelblue', 'coral']
colors_q = ['green', 'darkgreen']

for j in range(2):
    ax.quiver(*origin, *A[:, j], color=colors_a[j], arrow_length_ratio=0.08,
              linewidth=2.5, label=f'$a_{j+1}$')
    ax.quiver(*origin, *Q[:, j], color=colors_q[j], arrow_length_ratio=0.1,
              linewidth=2.5, linestyle='--', label=f'$q_{j+1}$')

ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_zlabel('$z$')
ax.set_title('Gram-Schmidt: original columns → orthonormal basis', fontsize=11)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()